# Evaluation on DeepSeek-LLM-7B  using FLORES+ benchmark for translation tasks

In [1]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.4 MB/s eta 0:00:00


In [2]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.1 MB/s eta 0:00:00


In [3]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.3 MB/s eta 0:00:00


In [4]:
import torch
from datasets import load_dataset
import pandas as pd
import sacrebleu
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from evaluate import load

In [5]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [6]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
ds_mkd = load_dataset("openlanguagedata/flores_plus", "mkd_Cyrl")

README.md:   0%|          | 0.00/73.9k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/224 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/219 [00:00<?, ?it/s]

mkd_Cyrl.jsonl:   0%|          | 0.00/535k [00:00<?, ?B/s]

mkd_Cyrl.jsonl:   0%|          | 0.00/559k [00:00<?, ?B/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

In [8]:
ds_eng = load_dataset("openlanguagedata/flores_plus", "eng_Latn")

Resolving data files:   0%|          | 0/224 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/219 [00:00<?, ?it/s]

eng_Latn.jsonl:   0%|          | 0.00/423k [00:00<?, ?B/s]

eng_Latn.jsonl:   0%|          | 0.00/440k [00:00<?, ?B/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

In [9]:
ds_mkd

DatasetDict({
    dev: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 997
    })
    devtest: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 1012
    })
})

In [10]:
ds_eng

DatasetDict({
    dev: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 997
    })
    devtest: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 1012
    })
})

In [11]:
dataset_mkd = ds_mkd['dev'].to_pandas()
dataset_mkd

,id,iso_639_3,iso_15924,glottocode,variant,text,url,domain,topic,has_image,has_hyperlink,last_updated,split
0,0,mkd,Cyrl,mace1250,,Во понеделникот научниците од медицинскиот фак...,https://en.wikinews.org/wiki/Scientists_say_ne...,wikinews,health,yes,yes,1.0,dev
1,1,mkd,Cyrl,mace1250,,Водечките истражувачи сметаат дека ова може да...,https://en.wikinews.org/wiki/Scientists_say_ne...,wikinews,health,yes,yes,1.0,dev
2,2,mkd,Cyrl,mace1250,,JAS 39C Грипен се сруши на писта околу 9:30 am...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
3,3,mkd,Cyrl,mace1250,,"Утврдено е дека пилот бил Дилокрит Патави, вод...",https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
4,4,mkd,Cyrl,mace1250,,Локалните медиуми известуваат за аеродромско п...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
...,...,...,...,...,...,...,...,...,...,...,...,...,...
992,992,mkd,Cyrl,mace1250,,Туристичката сезона на ридските станици во гла...,https://en.wikivoyage.org/wiki/Hill_stations_i...,wikivoyage,Natural wonders/Hill stations in India,yes,no,1.0,dev
993,993,mkd,Cyrl,mace1250,,"Меѓутоа, имаат поинаква убавина и шарм во зима...",https://en.wikivoyage.org/wiki/Hill_stations_i...,wikivoyage,Natural wonders/Hill stations in India,yes,no,1.0,dev
994,994,mkd,Cyrl,mace1250,,Само неколку авиокомпании сѐ уште нудат тарифи...,https://en.wikivoyage.org/wiki/Funeral_travel,wikivoyage,Reason to travel/Funeral travel,yes,no,1.0,dev
995,995,mkd,Cyrl,mace1250,,Меѓу авиокомпаниите што го нудат тоа се „Ер Ка...,https://en.wikivoyage.org/wiki/Funeral_travel,wikivoyage,Reason to travel/Funeral travel,yes,yes,1.0,dev


In [12]:
dataset_eng = ds_eng['dev'].to_pandas()
dataset_eng

,id,iso_639_3,iso_15924,glottocode,variant,text,url,domain,topic,has_image,has_hyperlink,last_updated,split
0,0,eng,Latn,stan1293,,"On Monday, scientists from the Stanford Univer...",https://en.wikinews.org/wiki/Scientists_say_ne...,wikinews,health,yes,yes,1.0,dev
1,1,eng,Latn,stan1293,,Lead researchers say this may bring early dete...,https://en.wikinews.org/wiki/Scientists_say_ne...,wikinews,health,yes,yes,1.0,dev
2,2,eng,Latn,stan1293,,The JAS 39C Gripen crashed onto a runway at ar...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
3,3,eng,Latn,stan1293,,The pilot was identified as Squadron Leader Di...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
4,4,eng,Latn,stan1293,,Local media reports an airport fire vehicle ro...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
...,...,...,...,...,...,...,...,...,...,...,...,...,...
992,992,eng,Latn,stan1293,,The tourist season for the hill stations gener...,https://en.wikivoyage.org/wiki/Hill_stations_i...,wikivoyage,Natural wonders/Hill stations in India,yes,no,1.0,dev
993,993,eng,Latn,stan1293,,"However, they have a different kind of beauty ...",https://en.wikivoyage.org/wiki/Hill_stations_i...,wikivoyage,Natural wonders/Hill stations in India,yes,no,1.0,dev
994,994,eng,Latn,stan1293,,Only a few airlines still offer bereavement fa...,https://en.wikivoyage.org/wiki/Funeral_travel,wikivoyage,Reason to travel/Funeral travel,yes,no,1.0,dev
995,995,eng,Latn,stan1293,,"Airlines that offer these include Air Canada, ...",https://en.wikivoyage.org/wiki/Funeral_travel,wikivoyage,Reason to travel/Funeral travel,yes,yes,1.0,dev


In [13]:
text_mkd = dataset_mkd['text'].values.tolist()[:100]
text_eng = dataset_eng['text'].values.tolist()[:100]

In [14]:
len(text_mkd)

100

In [15]:
len(text_eng)

100

In [16]:
text_mkd[:10]

['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.',
 'Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.',
 'JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.',
 'Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.',
 'Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.',
 '28-годишниот Вида

In [17]:
text_eng[:10]

['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.',
 'Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.',
 'The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.',
 'The pilot was identified as Squadron Leader Dilokrit Pattavee.',
 'Local media reports an airport fire vehicle rolled over while responding.',
 '28-year-old Vidal had joined Barça three seasons ago, from Sevilla.',
 'Since moving to the Catalan-capital, Vidal had played 49 games for the club.',
 "The protest started around 11:00 local time (UTC+1) on 

In [18]:
model_name = "deepseek-ai/deepseek-llm-7b-chat"

In [19]:
bitsandbytes_config = BitsAndBytesConfig(load_in_4bit=True,
                                         bnb_4bit_compute_dtype=torch.float16,
                                         bnb_4bit_quant_type='nf4',
                                         bnb_4bit_use_double_quant=True)

In [20]:
tokenizer = AutoTokenizer.from_pretrained(model_name,padding_side="left")
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             quantization_config=bitsandbytes_config,
                                             device_map="auto",
                                             offload_folder="offload")

config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [21]:
def get_prompt_mkd(instruction,text,language):
  return f"{instruction}\nТекст:{text}\nОдговор ({language}):"


In [22]:
def get_prompt_srb(instruction,text,language):
  return f"{instruction}\nТекст:{text}\nОдговор ({language}):"

In [23]:
def get_prompt_bug(instruction,text,language):
  return f"{instruction}\nТекст:{text}\nОтговор ({language}):"

In [24]:
def get_prompt_eng(instruction,text,language):
  return f"{instruction}\nText:{text}\nResponse ({language}):"

In [25]:
def batch_translate(texts,instruction, batch_size=8, max_new_tokens=256,lang_code="eng",language="English"):
    all_preds = []
    tokenizer.pad_token = tokenizer.eos_token
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        if lang_code=="mkd":
          prompts = [[{"role": "user", "content":get_prompt_mkd(instruction,t,language)}] for t in batch]
        elif lang_code == "bug":
          prompts = [[{"role": "user", "content":get_prompt_bug(instruction,t,language)}] for t in batch]
        elif lang_code == "srb":
          prompts = [[{"role": "user", "content":get_prompt_srb(instruction,t,language)}] for t in batch]
        else:
          prompts = [[{"role": "user", "content":get_prompt_eng(instruction,t,language)}] for t in batch]

        inputs = tokenizer.apply_chat_template(
            prompts,
            add_generation_prompt=True,
            tokenize=True,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.0,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        input_len = inputs.input_ids.shape[1]
        decoded = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)

        for translation in decoded:
            print(f"Batch {i}")
            clean_translation = translation.split("\n")[0].strip()
            all_preds.append(clean_translation)
            print(f"Result: {clean_translation}\n")

    return all_preds



In [26]:
from nltk.tokenize import sent_tokenize

def first_sentence(text):
    sents = sent_tokenize(text)
    return sents[0] if sents else text

In [27]:
bleu = load('bleu')

## Few-Shot Mono-lingual prompt (MKD->ENG)

In [28]:
instruction = f"""Преведи го следниот текст од **Македонски** во **Англиски**.Само испечати го преводот на англиски. Не додавај никакви објаснувања или дополнителни примери.
  Пример 1
  Македонски: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
  Одговор (Англиски): At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
  Пример 2
  Македонски: Еден спортски натпревар трае 90 минунти, односно две полувремиња по 45 минути со одмор меѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
  Одговор (Англиски): A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
  Пример 3
  Македонски: Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
  Одговор (Англиски): With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
  Пример 4
  Македонски: Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
  Одговор (Англиски): This study focuses on the development of new methods for data analysis.
  Пример 5
  Македонски: Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
  Одговор (Англиски): In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
"""

In [29]:
predictions_eng = batch_translate(text_mkd,instruction,max_new_tokens=128,lang_code="mkd",language="Англиски")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Batch 0
Result: On Monday, scientists from Stanford University's medical school announced the creation of a new tool for diagnosing using which cells are sorted according to type: work is underway for a tiny printer chip capable of being produced with the help of standard ink-jet printers, and it costs around one American cent.

Batch 0
Result: The leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV, and malaria among patients living in countries with low life expectancy, where the survival rate of diseases such as breast cancer may be halved compared to wealthy nations.

Batch 0
Result: The JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded due to the fact that the airport was closed for commercial flights.

Batch 0
Result: It is confirmed that the pilot was Diokletian Patavi, leader of the squadron.

Batch 0
Result: Local media reported about an aerial firefighting vehicle that overturned while firefight

In [30]:
references = [[ref] for ref in text_eng]
references[:10]

[['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.'],
 ['Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.'],
 ['The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.'],
 ['The pilot was identified as Squadron Leader Dilokrit Pattavee.'],
 ['Local media reports an airport fire vehicle rolled over while responding.'],
 ['28-year-old Vidal had joined Barça three seasons ago, from Sevilla.'],
 ['Since moving to the Catalan-capital, Vidal had played 49 games for the club.'],
 ["The protest started around 11:00 local t

In [31]:
bleu.compute(predictions=predictions_eng, references=references)

{'bleu': 0.23767732702527708,
 'precisions': [0.5601816805450416,
  0.2985837922895358,
  0.1773136773136773,
  0.107600341588386],
 'brevity_penalty': 1.0,
 'length_ratio': 1.055111821086262,
 'translation_length': 2642,
 'reference_length': 2504}

In [32]:
chrf_score = sacrebleu.corpus_chrf(predictions_eng, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 52.10


## Few-Shot Cross-lingual prompt (MKD->ENG) in english language

In [33]:
instruction = f"""Translate the following text from **Macedonian** to **English**. Only output the translation. Do not add explanations.
  Example 1
  Macedonian: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
  Response (English): At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
  Example 2
  Macedonian: Еден спортски натпревар трае 90 минунти, односно две полувремиња по 45 минути со одмор меѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
  Response (English): A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
  Example 3
  Macedonian: Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
  Response (English): With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
  Example 4
  Macedonian: Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
  Response (English): This study focuses on the development of new methods for data analysis.
  Example 5
  Macedonian: Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
  Response (English): In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
"""

In [34]:
predictions_eng = batch_translate(text_mkd,instruction,max_new_tokens=128)

Batch 0
Result: On Monday, scientists from Stanford University Medical School announced the creation of a new diagnostic tool that sorts cells based on their type. The focus is on a tiny printer chip capable of producing at around $1.

Batch 0
Result: The leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV, and malaria among patients living in countries with low life expectancy, where the survival rate of diseases such as breast cancer may be halved compared to wealthy nations.

Batch 0
Result: The JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded due to the fact that the airport was closed for commercial flights.

Batch 0
Result: It is confirmed that the pilot was Diokrit Patavi, leader of the squadron.

Batch 0
Result: Local media reports about an overturned firefighting aircraft at the airport while firefighters were responding to the call.

Batch 0
Result: A 28-year-old Vidal joins Barcelona's team.



In [35]:
references = [[ref] for ref in text_eng]
references[:10]

[['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.'],
 ['Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.'],
 ['The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.'],
 ['The pilot was identified as Squadron Leader Dilokrit Pattavee.'],
 ['Local media reports an airport fire vehicle rolled over while responding.'],
 ['28-year-old Vidal had joined Barça three seasons ago, from Sevilla.'],
 ['Since moving to the Catalan-capital, Vidal had played 49 games for the club.'],
 ["The protest started around 11:00 local t

In [36]:
bleu.compute(predictions=predictions_eng, references=references)

{'bleu': 0.2584475783788139,
 'precisions': [0.5886222910216719,
  0.32367149758454106,
  0.19379194630872484,
  0.12084063047285463],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0319488817891374,
 'translation_length': 2584,
 'reference_length': 2504}

In [37]:
chrf_score = sacrebleu.corpus_chrf(predictions_eng, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 52.11


## Few-Shot Cross-lingual prompt (MKD->ENG) in serbian language

In [38]:
instruction= f"""Преведи следећи текст са **Македонског** на **Енглески** језик.
Само испиши превод. Не додавај никаква објашњења.

Пример 1
Македонски: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
Одговор (Енглески): At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.

Пример 2
Македонски: Еден спортски натпревар трае 90 минути, односно две полувремиња по 45 минути со одмор помеѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
Одговор (Енглески): A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.

Пример 3
Македонски: Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
Одговор (Енглески): With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.

Пример 4
Македонски: Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
Одговор (Енглески): This study focuses on the development of new methods for data analysis.

Пример 5
Македонски: Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
Одговор (Енглески): In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
"""


In [39]:
predictions_eng = batch_translate(text_mkd,instruction,max_new_tokens=128,lang_code="srb",language="Енглески")

Batch 0
Result: On Monday, scientists from Stanford University's medical school announced the creation of a new tool for diagnosing using which cells are sorted according to type: work is being done on a tiny printer chip capable of printing standard inkjet printers, and it costs around $1.

Batch 0
Result: The lead researchers believe that this could lead to early detection of cancer, tuberculosis, HIV, and malaria among patients living in countries with low life standards, where the survival rate of diseases such as breast cancer can be halved compared to wealthy nations.

Batch 0
Result: The JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded due to the fact that the airport was closed for commercial flights.

Batch 0
Result: It has been confirmed that the pilot was Dioklit Patavi, leader of the squadron.

Batch 0
Result: Local media reports about an overturned firefighting aircraft at the airport.

Batch 0
Result: The 28-year-old Sevilla player jo

In [40]:
references = [[ref] for ref in text_eng]
references[:10]

[['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.'],
 ['Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.'],
 ['The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.'],
 ['The pilot was identified as Squadron Leader Dilokrit Pattavee.'],
 ['Local media reports an airport fire vehicle rolled over while responding.'],
 ['28-year-old Vidal had joined Barça three seasons ago, from Sevilla.'],
 ['Since moving to the Catalan-capital, Vidal had played 49 games for the club.'],
 ["The protest started around 11:00 local t

In [41]:
bleu.compute(predictions=predictions_eng, references=references)

{'bleu': 0.21744712142786288,
 'precisions': [0.5222861250898634,
  0.2759134973900075,
  0.1611154144074361,
  0.09629331184528606],
 'brevity_penalty': 1.0,
 'length_ratio': 1.1110223642172523,
 'translation_length': 2782,
 'reference_length': 2504}

In [42]:
chrf_score = sacrebleu.corpus_chrf(predictions_eng, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 54.08


## Few-Shot Cross-lingual prompt (MKD->ENG) in bulgarian language

In [43]:
instruction = f"""Преведи следния текст от **Македонски** на **Английски** език.
Само отпечатай превода на английски. Не добавяй никакви обяснения.

Пример 1
Македонски: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
Отговор (Английски): At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.

Пример 2
Македонски: Еден спортски натпревар трае 90 минути, односно две полувремиња по 45 минути со одмор помеѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
Отговор (Английски): A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.

Пример 3
Македонски: Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
Отговор (Английски): With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.

Пример 4
Македонски: Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
Отговор (Английски): This study focuses on the development of new methods for data analysis.

Пример 5
Македонски: Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
Отговор (Английски): In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
"""

In [44]:
predictions_eng = batch_translate(text_mkd,instruction,max_new_tokens=128,lang_code="bug",language="Английски")

Batch 0
Result: On Monday, scientists from Stanford University's medical school announced the creation of a new tool for diagnosing using which cells can be sorted according to type: work is being done on a tiny printer chip capable of being produced with the help of standard ink-jet printers, and it costs around one American cent.

Batch 0
Result: The lead researchers believe that this could lead to early detection of cancer, tuberculosis, HIV, and malaria among patients living in countries with low life standards, where the survival rate of diseases such as breast cancer may be halved compared to wealthy nations.

Batch 0
Result: The JAS 39C Gripen aircraft crashed on the runway around 9:30 am local time (2:30 UTC) and exploded due to the fact that the airport was closed for commercial flights.

Batch 0
Result: Detected that Diokrit Patavi was the leader of the squadron.

Batch 0
Result: Local media reported about an overturned firefighting aircraft at the airport while firefighters 

In [45]:
references = [[ref] for ref in text_eng]
references[:10]

[['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.'],
 ['Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.'],
 ['The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.'],
 ['The pilot was identified as Squadron Leader Dilokrit Pattavee.'],
 ['Local media reports an airport fire vehicle rolled over while responding.'],
 ['28-year-old Vidal had joined Barça three seasons ago, from Sevilla.'],
 ['Since moving to the Catalan-capital, Vidal had played 49 games for the club.'],
 ["The protest started around 11:00 local t

In [46]:
bleu.compute(predictions=predictions_eng, references=references)

{'bleu': 0.20687476718206202,
 'precisions': [0.49142661179698216,
  0.2571022727272727,
  0.15353460972017674,
  0.09441896024464831],
 'brevity_penalty': 1.0,
 'length_ratio': 1.1645367412140575,
 'translation_length': 2916,
 'reference_length': 2504}

In [47]:
chrf_score = sacrebleu.corpus_chrf(predictions_eng, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 52.74


## Few-Shot Mono-lingual prompt (ENG->MKD)

In [48]:
instruction = f"""Translate the following text from **English** into **Macedonian**. Only output the translation. Do not add explanations.
  Example 1
  English: On Monday, at 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
  Response (Macedonian): Во понеделник, во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
  Example 2
  English: A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
  Response (Macedonian): Еден спортски натпревар трае 90 минунти, односно две полувремиња по 45 минути со одмор меѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
  Example 3
  English: With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
  Response (Macedonian): Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
  Example 4
  English: This study focuses on the development of new methods for data analysis.
  Response (Macedonian)): Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
  Example 5
  English: In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
  Response (Macedonian): Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
"""

In [49]:
predictions_mkd = batch_translate(text_eng,instruction,max_new_tokens=128,language="Macedonian")

Batch 0
Result: Во понеделник, научниците од Калифорниската Универзитет за Медицински Наукови Центри обявиха варника на нов диагностичен инструмент кој може да сортира клетки според типот: малък принтерибилни щифт кој може да се произведат користејџ стандардни инклет печатници за вероватно еднаста по跌неска цента американски.

Batch 0
Result: Представници на водещи научни радниците го казват дека може да биде постигнато рано откривање на рак, туберкулоза, ХИВ и маларија за пациенти во бедна земја, каде преживјето на болестите као што е брестков рак може да биде половина од преживјето на ricsите земји.

Batch 0
Result: JAS 39C Гупен се спринтира на писта при околно 9:30 ч. локална врема (0230 UTC) и експлодира, затворивaа аеродромот за комунални полети.

Batch 0
Result: Полетната листа беше идентификована като Капитуларния лидер Дилоквит Паттави.

Batch 0
Result: Месачката медијска пропаганда го веома сериозно прикаже несреќен инцидент во авио-хибридската база во Прищина.

Batch 0
Result

In [50]:
references = [[ref] for ref in text_mkd]
references[:10]

[['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.'],
 ['Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.'],
 ['JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.'],
 ['Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.'],
 ['Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.'],
 ['28-год

In [51]:
bleu.compute(predictions=predictions_mkd, references=references)

{'bleu': 0.06095416399106672,
 'precisions': [0.3470636215334421,
  0.10034013605442177,
  0.03641207815275311,
  0.01254646840148699],
 'brevity_penalty': 0.9651408402912618,
 'length_ratio': 0.9657345411579362,
 'translation_length': 2452,
 'reference_length': 2539}

In [52]:
chrf_score = sacrebleu.corpus_chrf(predictions_mkd, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 45.07


## Few-Shot Cross-lingual prompt (ENG->MKD) in macedonian language

In [53]:
instruction = f"""Преведи го следниот текст од **Англиски** во **Македонски**. Само испечати го преводот на македонски јазик. Не додавај никакви објаснувања или дополнителни примери.
  Пример 1
  Англиски: On Monday, at 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
  Одговор (Македонски): Во понеделник, во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
  Пример 2
  Англиски: A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
  Одговор (Македонски): Еден спортски натпревар трае 90 минунти, односно две полувремиња по 45 минути со одмор меѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
  Пример 3
  Англиски: With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
  Одговор (Македонски): Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
  Пример 4
  Англиски: This study focuses on the development of new methods for data analysis.
  Одговор (Македонски): Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
  Пример 5
  Англиски: In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
  Одговор (Македонски): Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
"""

In [54]:
predictions_mkd = batch_translate(text_eng,instruction,max_new_tokens=128,lang_code="mkd",language="Македонски")

Batch 0
Result: Во понеделник, научниците од Калифорниската Универзитет за Медицински Наукови работи обявиха варника на нов диагностички инструмент кој е намерен за сортирање на клетки по вид: мален принтерисен чип кој може да се произвелзне со стандардни инкејт принтери за веројатно од од од од од од од од од од од од од од од од од од од од од од

Batch 0
Result: Главни исследователи кажу дека ова може да импресионира рано откривање на рака, туберкулоза, ХИВ и маларија на пациенти во лозни земји, каде преживотот нив за болестите као што е брестоCKS карактерише половина оние на ricsите земји.

Batch 0
Result: Јаз 39C Гупин се счукан на писта во околно 9:30 чортниот локален време (0230 UTC) и експлодираше, заclosingа аеродромот на трговинскиот летуви.

Batch 0
Result: Ќе се направи превод на англиската верзија текста во македонски јазик. Можете да ги чекате неколку секунди, бидејќи ќе се претходио тест заврши.

Batch 0
Result: Не можам да ти напиша отговор на Македонски, бидејќи немате

In [55]:
references = [[ref] for ref in text_mkd]
references[:10]

[['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.'],
 ['Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.'],
 ['JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.'],
 ['Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.'],
 ['Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.'],
 ['28-год

In [56]:
bleu.compute(predictions=predictions_mkd, references=references)

{'bleu': 0.03629787659786015,
 'precisions': [0.22490381252186079,
  0.04566872055092425,
  0.018804061677322303,
  0.008987885892926924],
 'brevity_penalty': 1.0,
 'length_ratio': 1.1260338716029934,
 'translation_length': 2859,
 'reference_length': 2539}

In [57]:
chrf_score = sacrebleu.corpus_chrf(predictions_mkd, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 41.13


## Few-Shot Cross-lingual prompt (ENG->MKD) in serbian language

In [58]:
instruction = f"""Преведи следећи текст са **Eнглеског** на **Mакедонски** језик.
Само испиши превод на македонски језик. Не додавај никаква објашњења или додатне примере.

Пример 1
Енглески: On Monday, at 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
Одговор (Mакедонски): Во понеделник, во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.

Пример 2
Енглески: A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
Одговор (Mакедонски): Еден спортски натпревар трае 90 минути, односно две полувремиња по 45 минути со одмор помеѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.

Пример 3
Енглески: With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
Одговор (Mакедонски): Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.

Пример 4
Енглески: This study focuses on the development of new methods for data analysis.
Одговор (Mакедонски): Оваа студија се фокусира на развојот на нови методи за анализа на податоци.

Пример 5
Енглески: In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
Одговор (Mакедонски): Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
"""

In [59]:
predictions_mkd = batch_translate(text_eng,instruction,max_new_tokens=128,lang_code="srb",language="Македонски")

Batch 0
Result: Во понеделник, научниците од Калифорниската Универзитет за Медицина откриле су новинска дијagnosticска средства која може да сортира клетки според типот: малка принтеримачка шепа која може да се произведу со стандардни инкејт принтери за веројатно од од 1 цента Амерички.

Batch 0
Result: Главни научници кажу дека може да донесе рано откривање откази на рака, туберкулоза, ХИВ и маларија на пациенти во ниски приходи земји, где преживотот нивите на болестите како мамкачки рак може биде половина оние на ricsите земји.

Batch 0
Result: Јаз 39C Гупен се сприел на писта во околно 9:30 ч. локална врема (0230 UTC) и експлодираше, заclosingа аеродромот за трговински летуви.

Batch 0
Result: Преведено на македонски: Приотлекуваната личност беше идентификована како Седнион Ледер Дилоквит Патрави.

Batch 0
Result: Локалните медијски вести гоReportingeше во неколку случаи на лошо функционирање на вознициот систем во мнозинството од земите во регион. Се повторно, овие случаи немаат ди

In [60]:
references = [[ref] for ref in text_mkd]
references[:10]

[['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.'],
 ['Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.'],
 ['JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.'],
 ['Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.'],
 ['Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.'],
 ['28-год

In [61]:
bleu.compute(predictions=predictions_mkd, references=references)

{'bleu': 0.05829979960995768,
 'precisions': [0.34568896051571313,
  0.09235936188077246,
  0.034618755477651184,
  0.011457378551787351],
 'brevity_penalty': 0.9772963459931288,
 'length_ratio': 0.9775502166207168,
 'translation_length': 2482,
 'reference_length': 2539}

In [62]:
chrf_score = sacrebleu.corpus_chrf(predictions_mkd, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 39.52


## Few-Shot Cross-lingual prompt (ENG->MKD) in bulgarian language

In [63]:
instruction = f"""Преведи следния текст от **Aнглийски** на **Mакедонски** език.
Само отпечатай превода на македонски език. Не добавяй никакви обяснения или допълнителни примери.

Пример 1
Английски: On Monday, at 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
Отговор (Mакедонски): Во понеделник, во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.

Пример 2
Английски: A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
Отговор (Mакедонски): Еден спортски натпревар трае 90 минути, односно две полувремиња по 45 минути со одмор помеѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.

Пример 3
Английски: With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
Отговор (Mакедонски): Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.

Пример 4
Английски: This study focuses on the development of new methods for data analysis.
Отговор (Mакедонски): Оваа студија се фокусира на развојот на нови методи за анализа на податоци.

Пример 5
Английски: In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
Отговор (Mакедонски): Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
"""


In [64]:
predictions_mkd = batch_translate(text_eng,instruction,max_new_tokens=128,lang_code="bug",language="Македонски")

Batch 0
Result: Во понеделно, научниците од Универзитетот на Станфорд попримачот Medicina ќе објави новото дијагностичко оруѓание, кое може да сортира клетки по врсту - малкo Printable шеички кој може да се произведу користење стандардни инкејт принтери за веројатно околу една Америчка цен.

Batch 0
Result: Представуваните учени кажу дека може би това ќе доведе рано откривање на рака, туберкулоза, ХИВ и маларија на пациентите во бедни земји, каде животестотата на болестите као што е брестеноста на мамка може да бидат половина оние на ricsите земји.

Batch 0
Result: JAS 39C Gripen се сруши на писта во околноста 9:30 утрината локална времина (0230 UTC) и експлодира, чинејќи аеродропот неприступачен за комунални полети.

Batch 0
Result: Преведено: Идентифичан е као Седничар Лидер Дилоктит Паттави.

Batch 0
Result: Локалните медији го зборуваат за пожарната комора која је скоканде во времешен испит.

Batch 0
Result: Валентин, който е на 28 години, е влездел во Барселона три сезона пред, од

In [65]:
references = [[ref] for ref in text_mkd]
references[:10]

[['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.'],
 ['Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.'],
 ['JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.'],
 ['Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.'],
 ['Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.'],
 ['28-год

In [66]:
bleu.compute(predictions=predictions_mkd, references=references)

{'bleu': 0.05505150429263826,
 'precisions': [0.3273260687342833,
  0.09011373578302712,
  0.03110704483074108,
  0.012937230474365118],
 'brevity_penalty': 0.9378886406630043,
 'length_ratio': 0.9397400551398188,
 'translation_length': 2386,
 'reference_length': 2539}

In [67]:
chrf_score = sacrebleu.corpus_chrf(predictions_mkd, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 39.69
